# Notebook 01: Core Types

**Purpose:** Develop and test fundamental types for ERPNext migration

**Contents:**
- Money class (currency, precision, validation)
- Account reference type
- Tax rate calculator
- Fiscal period validator

**Philosophy:** Build from primitives. Every higher-level type depends on these foundations.

## Setup

In [1]:
# Add src to path for development
import sys
from pathlib import Path

# Repository root
REPO_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
SRC_DIR = REPO_ROOT / 'src'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Repository root: {REPO_ROOT}")
print(f"Source directory: {SRC_DIR}")

Repository root: /home/jovyan/work/ERP/emt
Source directory: /home/jovyan/work/ERP/emt/src


In [2]:
# Standard library
from decimal import Decimal
from typing import List

# Third party
import pandas as pd

# Auto-reload modules during development
%load_ext autoreload
%autoreload 2

## Money Class

**Design goals:**
1. Immutable (thread-safe, prevents accidental modification)
2. Decimal precision (no float rounding errors)
3. Currency-aware (prevents mixing currencies)
4. ERPNext-compatible output format

**Why Decimal over float?**
```python
# Float precision issues
0.1 + 0.2 == 0.3  # False in Python!

# Decimal is exact
Decimal('0.1') + Decimal('0.2') == Decimal('0.3')  # True
```

In [3]:
from core.money import Money, get_currency_precision

### Test 1: Basic creation and validation

In [4]:
# Create money from different input types
m1 = Money(100, "KES")           # Integer
m2 = Money(100.50, "KES")        # Float
m3 = Money("100.50", "KES")      # String
m4 = Money(Decimal('100.50'), "KES")  # Decimal

print("Created money objects:")
print(f"From int:     {m1} = {m1.amount}")
print(f"From float:   {m2} = {m2.amount}")
print(f"From string:  {m3} = {m3.amount}")
print(f"From Decimal: {m4} = {m4.amount}")

assert m2 == m3 == m4, "Different input types should create equal Money"

Created money objects:
From int:     KES 100.00 = 100.00
From float:   KES 100.50 = 100.50
From string:  KES 100.50 = 100.50
From Decimal: KES 100.50 = 100.50


### Test 2: Rounding behavior

In [5]:
# Test rounding (ROUND_HALF_UP is standard accounting practice)
test_cases = [
    ("99.994", "99.99"),   # Round down
    ("99.995", "100.00"),  # Round up
    ("99.999", "100.00"),  # Round up
    ("0.005", "0.01"),     # Round up from exactly half
]

print("Rounding tests (2 decimal precision):")
for input_val, expected in test_cases:
    m = Money(input_val, "KES")
    result = str(m.amount)
    status = "✓" if result == expected else "✗"
    print(f"{status} {input_val:>10} → {result:>10} (expected {expected})")
    assert result == expected, f"Rounding failed for {input_val}"

Rounding tests (2 decimal precision):
✓     99.994 →      99.99 (expected 99.99)
✓     99.995 →     100.00 (expected 100.00)
✓     99.999 →     100.00 (expected 100.00)
✓      0.005 →       0.01 (expected 0.01)


### Test 3: Arithmetic operations

In [6]:
# Addition
m1 = Money(100, "KES")
m2 = Money(50.50, "KES")
m_sum = m1 + m2
print(f"Addition: {m1} + {m2} = {m_sum}")
assert m_sum.amount == Decimal('150.50')

# Subtraction
m_diff = m1 - m2
print(f"Subtraction: {m1} - {m2} = {m_diff}")
assert m_diff.amount == Decimal('49.50')

# Multiplication
m_mult = m1 * 1.5
print(f"Multiplication: {m1} × 1.5 = {m_mult}")
assert m_mult.amount == Decimal('150.00')

# Division
m_div = m1 / 4
print(f"Division: {m1} ÷ 4 = {m_div}")
assert m_div.amount == Decimal('25.00')

# Negation
m_neg = -m1
print(f"Negation: -{m1} = {m_neg}")
assert m_neg.amount == Decimal('-100.00')

print("\n✓ All arithmetic operations passed")

Addition: KES 100.00 + KES 50.50 = KES 150.50
Subtraction: KES 100.00 - KES 50.50 = KES 49.50
Multiplication: KES 100.00 × 1.5 = KES 150.00
Division: KES 100.00 ÷ 4 = KES 25.00
Negation: -KES 100.00 = KES -100.00

✓ All arithmetic operations passed


### Test 4: Currency mismatch protection

In [7]:
m_kes = Money(100, "KES")
m_usd = Money(100, "USD")

# Should raise ValueError
try:
    result = m_kes + m_usd
    print("✗ Should have raised ValueError for currency mismatch")
    assert False, "Currency mismatch not detected"
except ValueError as e:
    print(f"✓ Currency mismatch correctly detected: {e}")

# Comparison should also fail
try:
    result = m_kes > m_usd
    print("✗ Should have raised ValueError for currency comparison")
    assert False, "Currency comparison mismatch not detected"
except ValueError as e:
    print(f"✓ Currency comparison mismatch correctly detected: {e}")

✓ Currency mismatch correctly detected: Cannot add KES and USD
✓ Currency comparison mismatch correctly detected: Cannot compare KES with USD


### Test 5: ERPNext format compatibility

In [8]:
# ERPNext API expects float for currency fields
m = Money("123.456", "KES")  # Will round to 123.46

erpnext_value = m.to_erpnext_format()
print(f"Money object: {m}")
print(f"ERPNext format: {erpnext_value}")
print(f"Type: {type(erpnext_value)}")

assert isinstance(erpnext_value, float), "ERPNext format should be float"
assert erpnext_value == 123.46, "Value should be rounded correctly"

print("\n✓ ERPNext format conversion passed")

Money object: KES 123.46
ERPNext format: 123.46
Type: <class 'float'>

✓ ERPNext format conversion passed


### Test 6: Edge cases

In [9]:
# Zero
m_zero = Money.zero("KES")
print(f"Zero: {m_zero}, is_zero={m_zero.is_zero()}")
assert m_zero.is_zero()

# Negative
m_negative = Money(-50, "KES")
print(f"Negative: {m_negative}, is_negative={m_negative.is_negative()}")
assert m_negative.is_negative()
assert not m_negative.is_positive()

# From ERPNext None value
m_from_none = Money.from_erpnext(None, "KES")
print(f"From None: {m_from_none}")
assert m_from_none.is_zero()

# Division by zero should raise
try:
    m = Money(100, "KES") / 0
    print("✗ Should have raised ZeroDivisionError")
    assert False
except ZeroDivisionError:
    print("✓ Division by zero correctly raises error")

print("\n✓ All edge cases handled correctly")

Zero: KES 0.00, is_zero=True
Negative: KES -50.00, is_negative=True
From None: KES 0.00
✓ Division by zero correctly raises error

✓ All edge cases handled correctly


### Test 7: Real-world scenario - Invoice totals

In [10]:
# Simulate calculating invoice total from line items
line_items = [
    {"description": "Event Venue Hire", "amount": Money(15000, "KES")},
    {"description": "Room Accommodation (2 nights)", "amount": Money(14000, "KES")},
    {"description": "Wellness Session", "amount": Money(5000, "KES")},
]

# Calculate subtotal
subtotal = sum((item["amount"] for item in line_items), Money.zero("KES"))
print(f"Subtotal: {subtotal}")

# Apply tax (16% VAT in Kenya)
tax_rate = Decimal('0.16')
tax_amount = subtotal * tax_rate
print(f"Tax (16%): {tax_amount}")

# Calculate total
total = subtotal + tax_amount
print(f"Total: {total}")

# Verify calculations
assert subtotal.amount == Decimal('34000.00')
assert tax_amount.amount == Decimal('5440.00')
assert total.amount == Decimal('39440.00')

print("\n✓ Invoice calculation test passed")

# Show ERPNext API payload format
api_payload = {
    "doctype": "Sales Invoice",
    "total": subtotal.to_erpnext_format(),
    "total_taxes_and_charges": tax_amount.to_erpnext_format(),
    "grand_total": total.to_erpnext_format(),
}

print("\nERPNext API payload:")
for key, value in api_payload.items():
    if key != "doctype":
        print(f"  {key}: {value}")

Subtotal: KES 34,000.00
Tax (16%): KES 5,440.00
Total: KES 39,440.00

✓ Invoice calculation test passed

ERPNext API payload:
  total: 34000.0
  total_taxes_and_charges: 5440.0
  grand_total: 39440.0


## Summary

**Money class validation complete:**
- ✓ Creates from int, float, string, Decimal
- ✓ Rounds correctly (ROUND_HALF_UP)
- ✓ Arithmetic operations work
- ✓ Currency mismatch protection
- ✓ ERPNext format conversion
- ✓ Edge cases handled
- ✓ Real-world invoice scenario

**Next steps:**
1. Extract Money class to `src/core/money.py` ✓ (already done)
2. Create unit tests in `tests/test_core/test_money.py`
3. Develop Account type
4. Develop Tax type
5. Develop FiscalPeriod type

## Next: Account Type

**Requirements:**
- Account code validation (format checking)
- Account type validation (Asset, Liability, Income, Expense, Equity)
- Account hierarchy support (parent-child relationships)
- ERPNext Chart of Accounts compatibility

**To be developed in next cell...**

## Account Class

**Design goals:**
1. Represent Chart of Accounts entries
2. Validate account types (Asset, Liability, Income, Expense, Equity)
3. Support ERPNext naming convention (Name - Company)
4. Determine debit/credit behavior by account type

**ERPNext account naming:**
```
"Cash - WC"              ← Full name with company suffix
"Event Venue Hire - WC"  ← Revenue account
"Debtors - WC"           ← Accounts Receivable
```

### Load Account class

In [11]:
from core.account import Account, AccountType, parse_account_name, create_standard_accounts

### Test 1: Basic creation and formatting

In [13]:
# Create with full name
acc1 = Account("Cash - WC", AccountType.CASH)
print(f"Full name: {acc1.name}")
print(f"Base name: {acc1.base_name}")
print(f"Company: {acc1.company_suffix}")
print(f"Type: {acc1.account_type}")

print("\n" + "="*50 + "\n")

# Create with auto-formatting
acc2 = Account("Debtors", AccountType.RECEIVABLE, company="WC")
print(f"Auto-formatted: {acc2.name}")
print(f"Base name: {acc2.base_name}")
print(f"Company: {acc2.company_suffix}")

assert acc2.name == "Debtors - WC", "Company suffix should be appended"
assert acc2.base_name == "Debtors", "Base name extraction failed"
assert acc2.company_suffix == "WC", "Company suffix extraction failed"

print("\n✓ Account creation and formatting tests passed")

Full name: Cash - WC
Base name: Cash
Company: WC
Type: AccountType.CASH


Auto-formatted: Debtors - WC
Base name: Debtors
Company: WC

✓ Account creation and formatting tests passed


### Test 2: Account types and debit/credit behavior

In [14]:
# Test debit-positive accounts (Assets, Expenses)
asset_accounts = [
    Account("Cash - WC", AccountType.CASH),
    Account("Bank - WC", AccountType.BANK),
    Account("Debtors - WC", AccountType.RECEIVABLE),
    Account("Stock - WC", AccountType.STOCK),
]

expense_account = Account("Utilities - WC", AccountType.EXPENSE)

print("Debit-positive accounts (debits increase balance):")
for acc in asset_accounts + [expense_account]:
    assert acc.is_debit_positive(), f"{acc.name} should be debit-positive"
    print(f"  ✓ {acc.name} ({acc.account_type.value})")

print("\n" + "="*50 + "\n")

# Test credit-positive accounts (Liabilities, Equity, Income)
credit_accounts = [
    Account("Creditors - WC", AccountType.PAYABLE),
    Account("Capital Stock - WC", AccountType.EQUITY),
    Account("Event Venue Hire - WC", AccountType.INCOME),
]

print("Credit-positive accounts (credits increase balance):")
for acc in credit_accounts:
    assert not acc.is_debit_positive(), f"{acc.name} should be credit-positive"
    print(f"  ✓ {acc.name} ({acc.account_type.value})")

print("\n✓ Debit/credit behavior tests passed")

Debit-positive accounts (debits increase balance):
  ✓ Cash - WC (Cash)
  ✓ Bank - WC (Bank)
  ✓ Debtors - WC (Receivable)
  ✓ Stock - WC (Stock)
  ✓ Utilities - WC (Expense)


Credit-positive accounts (credits increase balance):
  ✓ Creditors - WC (Payable)
  ✓ Capital Stock - WC (Equity)
  ✓ Event Venue Hire - WC (Income)

✓ Debit/credit behavior tests passed


### Test 3: Balance Sheet vs P&L classification

In [15]:
# Balance Sheet accounts
balance_sheet = [
    Account("Cash - WC", AccountType.CASH),
    Account("Debtors - WC", AccountType.RECEIVABLE),
    Account("Creditors - WC", AccountType.PAYABLE),
    Account("Capital Stock - WC", AccountType.EQUITY),
]

print("Balance Sheet accounts:")
for acc in balance_sheet:
    assert acc.is_balance_sheet_account(), f"{acc.name} should be on Balance Sheet"
    assert not acc.is_profit_and_loss_account(), f"{acc.name} should not be on P&L"
    print(f"  ✓ {acc.name}")

print("\n" + "="*50 + "\n")

# P&L accounts
pnl_accounts = [
    Account("Event Venue Hire - WC", AccountType.INCOME),
    Account("Utilities - WC", AccountType.EXPENSE),
    Account("Salaries - WC", AccountType.EXPENSE),
]

print("Profit & Loss accounts:")
for acc in pnl_accounts:
    assert acc.is_profit_and_loss_account(), f"{acc.name} should be on P&L"
    assert not acc.is_balance_sheet_account(), f"{acc.name} should not be on Balance Sheet"
    print(f"  ✓ {acc.name}")

print("\n✓ Financial statement classification tests passed")

Balance Sheet accounts:
  ✓ Cash - WC
  ✓ Debtors - WC
  ✓ Creditors - WC
  ✓ Capital Stock - WC


Profit & Loss accounts:
  ✓ Event Venue Hire - WC
  ✓ Utilities - WC
  ✓ Salaries - WC

✓ Financial statement classification tests passed


### Test 4: Account hierarchy

In [16]:
# Parent-child relationships
parent = Account("Indirect Expenses - WC", AccountType.EXPENSE)
child = Account("Utilities - WC", AccountType.EXPENSE, parent_account=parent.name)

print(f"Parent account: {parent.name}")
print(f"Child account: {child.name}")
print(f"Child's parent: {child.parent_account}")

assert child.parent_account == parent.name, "Parent reference not stored correctly"

print("\n✓ Account hierarchy test passed")

Parent account: Indirect Expenses - WC
Child account: Utilities - WC
Child's parent: Indirect Expenses - WC

✓ Account hierarchy test passed


### Test 5: Account numbers (optional codes)

In [17]:
# Some organizations use account numbers for sorting
acc_with_number = Account(
    "Cash - WC",
    AccountType.CASH,
    account_number="1001"
)

print(f"Account: {acc_with_number}")
print(f"Number: {acc_with_number.account_number}")
print(f"String representation: {str(acc_with_number)}")

assert "1001" in str(acc_with_number), "Account number not in string format"

print("\n✓ Account number test passed")

Account: 1001 - Cash - WC
Number: 1001
String representation: 1001 - Cash - WC

✓ Account number test passed


### Test 6: ERPNext format conversion

In [18]:
acc = Account("Cash - WC", AccountType.CASH, account_number="1001")
erpnext_payload = acc.to_erpnext_format()

print("ERPNext API payload:")
for key, value in erpnext_payload.items():
    print(f"  {key}: {value}")

expected_fields = ["doctype", "account_name", "account_type", "company", "account_number"]
for field in expected_fields:
    assert field in erpnext_payload, f"Missing field: {field}"

assert erpnext_payload["account_name"] == "Cash", "Base name not extracted"
assert erpnext_payload["company"] == "WC", "Company not extracted"
assert erpnext_payload["account_type"] == "Cash", "Account type not converted"

print("\n✓ ERPNext format conversion test passed")

ERPNext API payload:
  doctype: Account
  account_name: Cash
  account_type: Cash
  account_number: 1001
  company: WC

✓ ERPNext format conversion test passed


### Test 7: Standard account creation helper

In [19]:
# Helper function for creating common accounts
standard_accounts = create_standard_accounts("WC")

print("Standard accounts created:")
for purpose, acc in standard_accounts.items():
    print(f"  {purpose:20} → {acc.name:30} ({acc.account_type.value})")

# Verify key accounts exist
assert "cash" in standard_accounts
assert "debtors" in standard_accounts
assert "revenue" in standard_accounts
assert "expenses" in standard_accounts

# Verify all have company suffix
for acc in standard_accounts.values():
    assert acc.company_suffix == "WC", f"{acc.name} missing company suffix"

print("\n✓ Standard accounts helper test passed")

Standard accounts created:
  cash                 → Cash - WC                      (Cash)
  bank                 → Bank - WC                      (Bank)
  debtors              → Debtors - WC                   (Receivable)
  stock                → Stock - WC                     (Stock)
  creditors            → Creditors - WC                 (Payable)
  capital              → Capital Stock - WC             (Equity)
  retained_earnings    → Retained Earnings - WC         (Equity)
  revenue              → Sales - WC                     (Income)
  expenses             → Expenses - WC                  (Expense)

✓ Standard accounts helper test passed


### Test 8: Parse account name utility

In [20]:
# Helper for extracting base name and company
test_cases = [
    ("Cash - WC", "Cash", "WC"),
    ("Event Venue Hire - WC", "Event Venue Hire", "WC"),
    ("Cash", "Cash", None),  # No suffix
]

print("Account name parsing:")
for full_name, expected_base, expected_company in test_cases:
    base, company = parse_account_name(full_name)
    print(f"  {full_name:30} → base='{base}', company='{company}'")
    assert base == expected_base, f"Base name mismatch for {full_name}"
    assert company == expected_company, f"Company mismatch for {full_name}"

print("\n✓ Parse account name test passed")

Account name parsing:
  Cash - WC                      → base='Cash', company='WC'
  Event Venue Hire - WC          → base='Event Venue Hire', company='WC'
  Cash                           → base='Cash', company='None'

✓ Parse account name test passed


### Test 9: Real-world scenario - Wellness Centre accounts

In [21]:
# Create accounts matching wellness centre's Chart of Accounts
wellness_accounts = {
    # Assets
    "mobile_money": Account("Mobile Money - WC", AccountType.BANK),
    "kcb_bank": Account("KCB - WC", AccountType.BANK),
    "cash": Account("Cash - WC", AccountType.CASH),
    "debtors": Account("Debtors - WC", AccountType.RECEIVABLE),
    
    # Liabilities
    "creditors": Account("Creditors - WC", AccountType.PAYABLE),
    
    # Equity
    "capital": Account("Capital Stock - WC", AccountType.EQUITY),
    
    # Income
    "event_venue": Account("Event Venue Hire - WC", AccountType.INCOME),
    "room_income": Account("Room Accommodation - WC", AccountType.INCOME),
    "egg_sales": Account("Farm Eggs - WC", AccountType.INCOME),
    "wellness_services": Account("Wellness Services - WC", AccountType.INCOME),
    
    # Expenses
    "utilities": Account("Utilities - WC", AccountType.EXPENSE, 
                        parent_account="Indirect Expenses - WC"),
    "salaries": Account("Salaries - WC", AccountType.EXPENSE,
                       parent_account="Direct Expenses - WC"),
    "inventory_purchases": Account("Inventory Purchases - WC", AccountType.EXPENSE,
                                  parent_account="Direct Expenses - WC"),
}

print("Wellness Centre Chart of Accounts:")
print("\nAssets:")
for name, acc in wellness_accounts.items():
    if acc.account_type in (AccountType.CASH, AccountType.BANK, AccountType.RECEIVABLE):
        print(f"  {acc.name}")

print("\nIncome:")
for name, acc in wellness_accounts.items():
    if acc.account_type == AccountType.INCOME:
        print(f"  {acc.name}")

print("\nExpenses:")
for name, acc in wellness_accounts.items():
    if acc.account_type == AccountType.EXPENSE:
        parent_info = f" (parent: {acc.parent_account})" if acc.parent_account else ""
        print(f"  {acc.name}{parent_info}")

# Verify all accounts have proper company suffix
for acc in wellness_accounts.values():
    assert acc.company_suffix == "WC", f"{acc.name} missing suffix"

# Verify debit/credit behavior for sample transaction
# "Received KES 15,000 event deposit via M-Pesa"
mobile_money = wellness_accounts["mobile_money"]
event_venue = wellness_accounts["event_venue"]

print(f"\nSample transaction verification:")
print(f"  Debit: {mobile_money.name} (increases? {mobile_money.is_debit_positive()})")
print(f"  Credit: {event_venue.name} (increases? {not event_venue.is_debit_positive()})")

assert mobile_money.is_debit_positive(), "Bank account should increase with debits"
assert not event_venue.is_debit_positive(), "Income should increase with credits"

print("\n✓ Wellness Centre accounts test passed")

Wellness Centre Chart of Accounts:

Assets:
  Mobile Money - WC
  KCB - WC
  Cash - WC
  Debtors - WC

Income:
  Event Venue Hire - WC
  Room Accommodation - WC
  Farm Eggs - WC
  Wellness Services - WC

Expenses:
  Utilities - WC (parent: Indirect Expenses - WC)
  Salaries - WC (parent: Direct Expenses - WC)
  Inventory Purchases - WC (parent: Direct Expenses - WC)

Sample transaction verification:
  Debit: Mobile Money - WC (increases? True)
  Credit: Event Venue Hire - WC (increases? True)

✓ Wellness Centre accounts test passed


## Summary - Account Class

**Account class validation complete:**
- ✓ Creates from full or base name
- ✓ Validates account types
- ✓ Extracts base name and company suffix
- ✓ Determines debit/credit behavior
- ✓ Classifies Balance Sheet vs P&L
- ✓ Supports account hierarchy
- ✓ Converts to ERPNext format
- ✓ Real-world Chart of Accounts tested

**Next:** Tax and FiscalPeriod types

## Tax Class

**Design goals:**
1. Represent tax rates with proper rounding
2. Calculate tax from base amounts
3. Extract tax from gross amounts
4. Support multiple tax types (VAT, withholding, etc.)

**Kenya tax context:**
- Standard VAT: 16%
- Withholding tax: 5% (resident contractors)
- Zero-rated goods: 0% VAT

### Load Tax classes

In [22]:
from core.tax import TaxRate, TaxType, create_kenya_tax_rates, calculate_tax_breakdown

### Test 1: Create tax rates

In [23]:
# Standard VAT in Kenya
vat = TaxRate(Decimal('0.16'), "VAT @ 16%", TaxType.VAT)
print(f"Tax rate: {vat}")
print(f"Rate: {vat.rate}")
print(f"Percentage: {vat.percentage}%")
print(f"Type: {vat.tax_type.value}")

assert vat.rate == Decimal('0.16'), "Rate should be stored as decimal"
assert vat.percentage == Decimal('16.00'), "Percentage conversion failed"

print("\n✓ Tax rate creation test passed")

Tax rate: VAT @ 16% (16.00%)
Rate: 0.16
Percentage: 16.00%
Type: VAT

✓ Tax rate creation test passed


### Test 2: Calculate tax from base

In [24]:
# Calculate 16% VAT on KES 10,000
vat = TaxRate(Decimal('0.16'), "VAT @ 16%")
base = Money(10000, "KES")

tax_amount = vat.calculate_tax(base)
total = vat.calculate_total(base)

print(f"Base amount: {base}")
print(f"Tax (16%): {tax_amount}")
print(f"Total: {total}")

assert tax_amount.amount == Decimal('1600.00'), "Tax calculation incorrect"
assert total.amount == Decimal('11600.00'), "Total calculation incorrect"

print("\n✓ Tax calculation test passed")

Base amount: KES 10,000.00
Tax (16%): KES 1,600.00
Total: KES 11,600.00

✓ Tax calculation test passed


### Test 3: Extract tax from gross amount

In [25]:
# Extract tax from total when base is unknown
# Useful for receipts that show only total
vat = TaxRate(Decimal('0.16'), "VAT @ 16%")
gross = Money(11600, "KES")

extracted_tax = vat.extract_tax(gross)
extracted_base = vat.extract_base(gross)

print(f"Gross amount: {gross}")
print(f"Extracted tax: {extracted_tax}")
print(f"Extracted base: {extracted_base}")

# Verify extraction is correct
assert extracted_tax.amount == Decimal('1600.00'), "Tax extraction incorrect"
assert extracted_base.amount == Decimal('10000.00'), "Base extraction incorrect"
assert extracted_base + extracted_tax == gross, "Components don't sum to gross"

print("\n✓ Tax extraction test passed")

Gross amount: KES 11,600.00
Extracted tax: KES 1,600.00
Extracted base: KES 10,000.00

✓ Tax extraction test passed


### Test 4: Create tax rate from percentage

In [26]:
# Convenience method for percentage input
vat = TaxRate.from_percentage(16, "VAT @ 16%")

print(f"Created from percentage: {vat}")
print(f"Rate: {vat.rate}")
print(f"Percentage: {vat.percentage}%")

assert vat.rate == Decimal('0.16'), "Percentage conversion failed"

print("\n✓ From percentage test passed")

Created from percentage: VAT @ 16% (16.00%)
Rate: 0.16
Percentage: 16.00%

✓ From percentage test passed


### Test 5: Zero-rated tax

In [27]:
# Some goods/services are zero-rated (0% tax)
zero_vat = TaxRate.zero_rated("VAT @ 0% (Zero Rated)")

base = Money(5000, "KES")
tax = zero_vat.calculate_tax(base)
total = zero_vat.calculate_total(base)

print(f"Zero-rated tax: {zero_vat}")
print(f"Base: {base}")
print(f"Tax: {tax}")
print(f"Total: {total}")

assert tax.is_zero(), "Zero-rated tax should produce zero tax"
assert total == base, "Total should equal base for zero-rated"
assert zero_vat.is_zero_rated(), "Should identify as zero-rated"

print("\n✓ Zero-rated tax test passed")

Zero-rated tax: VAT @ 0% (Zero Rated) (0.00%)
Base: KES 5,000.00
Tax: KES 0.00
Total: KES 5,000.00

✓ Zero-rated tax test passed


### Test 6: Multiple taxes (tax breakdown)

In [28]:
# Calculate multiple taxes on same base
base = Money(10000, "KES")
vat = TaxRate(Decimal('0.16'), "VAT @ 16%")
service_tax = TaxRate(Decimal('0.02'), "Service Tax @ 2%")

breakdown = calculate_tax_breakdown(base, [vat, service_tax])

print(f"Base amount: {breakdown['base']}")
print("\nTaxes:")
for desc, amount in breakdown['taxes'].items():
    print(f"  {desc}: {amount}")
print(f"\nTotal tax: {breakdown['total_tax']}")
print(f"Grand total: {breakdown['total']}")

assert breakdown['total_tax'].amount == Decimal('1800.00'), "Multiple tax sum incorrect"
assert breakdown['total'].amount == Decimal('11800.00'), "Grand total incorrect"

print("\n✓ Multiple taxes test passed")

Base amount: KES 10,000.00

Taxes:
  VAT @ 16%: KES 1,600.00
  Service Tax @ 2%: KES 200.00

Total tax: KES 1,800.00
Grand total: KES 11,800.00

✓ Multiple taxes test passed


### Test 7: Kenya standard tax rates

In [29]:
# Pre-configured Kenya tax rates
kenya_taxes = create_kenya_tax_rates("WC")

print("Kenya standard tax rates:")
for purpose, tax_rate in kenya_taxes.items():
    print(f"  {purpose:25} → {tax_rate}")

# Test VAT calculation
vat = kenya_taxes["vat_standard"]
base = Money(15000, "KES")
total = vat.calculate_total(base)

print(f"\nVAT calculation example:")
print(f"  Base: {base}")
print(f"  Total with VAT: {total}")

assert vat.rate == Decimal('0.16'), "Kenya VAT rate incorrect"
assert total.amount == Decimal('17400.00'), "VAT calculation failed"

print("\n✓ Kenya tax rates test passed")

Kenya standard tax rates:
  vat_standard              → VAT @ 16% (16.00%)
  vat_zero                  → VAT @ 0% (Zero Rated) (0.00%)
  withholding_resident      → Withholding Tax @ 5% (Resident) (5.00%)
  withholding_professional  → Withholding Tax @ 5% (Professional Services) (5.00%)

VAT calculation example:
  Base: KES 15,000.00
  Total with VAT: KES 17,400.00

✓ Kenya tax rates test passed


### Test 8: ERPNext format conversion

In [30]:
vat = TaxRate(
    Decimal('0.16'),
    "VAT @ 16%",
    TaxType.VAT,
    "VAT - WC"
)

erpnext_payload = vat.to_erpnext_format()

print("ERPNext tax format:")
for key, value in erpnext_payload.items():
    print(f"  {key}: {value}")

assert erpnext_payload["charge_type"] == "On Net Total"
assert erpnext_payload["rate"] == 16.0, "Rate should be percentage"
assert erpnext_payload["account_head"] == "VAT - WC"

print("\n✓ ERPNext format test passed")

ERPNext tax format:
  charge_type: On Net Total
  description: VAT @ 16%
  rate: 16.0
  account_head: VAT - WC

✓ ERPNext format test passed


### Test 9: Real-world invoice calculation

In [31]:
# Wellness Centre invoice with VAT
line_items = [
    {"description": "Event Venue Hire", "amount": Money(15000, "KES")},
    {"description": "Room Accommodation", "amount": Money(14000, "KES")},
]

# Calculate subtotal
subtotal = sum((item["amount"] for item in line_items), Money.zero("KES"))

# Apply 16% VAT
vat = TaxRate(Decimal('0.16'), "VAT @ 16%", TaxType.VAT, "VAT - WC")
tax_amount = vat.calculate_tax(subtotal)
grand_total = subtotal + tax_amount

print("Invoice calculation:")
for item in line_items:
    print(f"  {item['description']:25} {item['amount']}")
print(f"  {'─' * 50}")
print(f"  {'Subtotal':25} {subtotal}")
print(f"  {vat.description:25} {tax_amount}")
print(f"  {'─' * 50}")
print(f"  {'Grand Total':25} {grand_total}")

assert subtotal.amount == Decimal('29000.00')
assert tax_amount.amount == Decimal('4640.00')
assert grand_total.amount == Decimal('33640.00')

print("\n✓ Real-world invoice test passed")

Invoice calculation:
  Event Venue Hire          KES 15,000.00
  Room Accommodation        KES 14,000.00
  ──────────────────────────────────────────────────
  Subtotal                  KES 29,000.00
  VAT @ 16%                 KES 4,640.00
  ──────────────────────────────────────────────────
  Grand Total               KES 33,640.00

✓ Real-world invoice test passed


### Test 10: Rate validation

In [32]:
# Test error handling
test_cases = [
    (Decimal('-0.05'), "Negative rate"),  # Should raise
    (Decimal('1.5'), "Rate > 1"),         # Should raise (150% entered as 1.5)
]

print("Validation tests:")
for rate, description in test_cases:
    try:
        invalid = TaxRate(rate, description)
        print(f"  ✗ {description}: Should have raised ValueError")
        assert False, f"Validation failed for {description}"
    except ValueError as e:
        print(f"  ✓ {description}: Correctly rejected")

print("\n✓ Rate validation test passed")

Validation tests:
  ✓ Negative rate: Correctly rejected
  ✓ Rate > 1: Correctly rejected

✓ Rate validation test passed


## Summary - Tax Class

**Tax class validation complete:**
- ✓ Creates from decimal or percentage
- ✓ Calculates tax from base amount
- ✓ Extracts tax from gross amount
- ✓ Supports zero-rated taxes
- ✓ Handles multiple taxes
- ✓ Kenya-specific tax rates
- ✓ ERPNext format conversion
- ✓ Real-world invoice calculations
- ✓ Proper validation

## FiscalPeriod Class

**Design goals:**
1. Represent accounting periods (year, quarter, month)
2. Validate date ranges
3. Check if dates fall within periods
4. Support ERPNext fiscal year setup

**Wellness Centre context:**
- Migration data: March 2024 - February 2025
- Requires: FY 2024, FY 2025

### Load FiscalPeriod classes

In [34]:
from core.fiscal_period import FiscalPeriod, PeriodType, create_fiscal_years, get_period_for_date
from datetime import date

### Test 1: Create fiscal year

In [35]:
# Standard calendar year
fy_2024 = FiscalPeriod.year(2024)

print(f"Fiscal year: {fy_2024}")
print(f"Name: {fy_2024.name}")
print(f"Start: {fy_2024.start_date}")
print(f"End: {fy_2024.end_date}")
print(f"Duration: {fy_2024.duration_days} days")
print(f"Type: {fy_2024.period_type.value}")

assert fy_2024.start_date == date(2024, 1, 1)
assert fy_2024.end_date == date(2024, 12, 31)
assert fy_2024.duration_days == 366  # 2024 is leap year

print("\n✓ Fiscal year creation test passed")

Fiscal year: FY 2024 (2024-01-01 to 2024-12-31)
Name: FY 2024
Start: 2024-01-01
End: 2024-12-31
Duration: 366 days
Type: Year

✓ Fiscal year creation test passed


### Test 2: Check date containment

In [36]:
fy_2024 = FiscalPeriod.year(2024)

test_dates = [
    (date(2024, 1, 1), True, "Start date"),
    (date(2024, 6, 15), True, "Mid-year"),
    (date(2024, 12, 31), True, "End date"),
    (date(2023, 12, 31), False, "Before start"),
    (date(2025, 1, 1), False, "After end"),
]

print("Date containment tests:")
for test_date, expected, description in test_dates:
    result = fy_2024.contains(test_date)
    status = "✓" if result == expected else "✗"
    print(f"  {status} {description}: {test_date} → {result}")
    assert result == expected, f"Containment check failed for {description}"

print("\n✓ Date containment test passed")

Date containment tests:
  ✓ Start date: 2024-01-01 → True
  ✓ Mid-year: 2024-06-15 → True
  ✓ End date: 2024-12-31 → True
  ✓ Before start: 2023-12-31 → False
  ✓ After end: 2025-01-01 → False

✓ Date containment test passed


### Test 3: Create quarters

In [37]:
# Q1-Q4 of 2024
quarters = [FiscalPeriod.quarter(2024, q) for q in range(1, 5)]

print("2024 Quarters:")
for q in quarters:
    print(f"  {q.name:10} {q.start_date} to {q.end_date} ({q.duration_days} days)")

# Verify Q1
q1 = quarters[0]
assert q1.start_date == date(2024, 1, 1)
assert q1.end_date == date(2024, 3, 31)
assert q1.name == "Q1 2024"

# Verify Q4
q4 = quarters[3]
assert q4.start_date == date(2024, 10, 1)
assert q4.end_date == date(2024, 12, 31)

print("\n✓ Quarter creation test passed")

2024 Quarters:
  Q1 2024    2024-01-01 to 2024-03-31 (91 days)
  Q2 2024    2024-04-01 to 2024-06-30 (91 days)
  Q3 2024    2024-07-01 to 2024-09-30 (92 days)
  Q4 2024    2024-10-01 to 2024-12-31 (92 days)

✓ Quarter creation test passed


### Test 4: Create months

In [38]:
# First quarter months
months = [FiscalPeriod.month(2024, m) for m in range(1, 4)]

print("Q1 2024 Months:")
for month in months:
    print(f"  {month.name:15} {month.start_date} to {month.end_date} ({month.duration_days} days)")

# Verify January
jan = months[0]
assert jan.start_date == date(2024, 1, 1)
assert jan.end_date == date(2024, 1, 31)
assert jan.duration_days == 31

# Verify February (leap year)
feb = months[1]
assert feb.duration_days == 29  # 2024 is leap year

print("\n✓ Month creation test passed")

Q1 2024 Months:
  January 2024    2024-01-01 to 2024-01-31 (31 days)
  February 2024   2024-02-01 to 2024-02-29 (29 days)
  March 2024      2024-03-01 to 2024-03-31 (31 days)

✓ Month creation test passed


### Test 5: Period overlap detection

In [39]:
fy_2024 = FiscalPeriod.year(2024)
fy_2025 = FiscalPeriod.year(2025)
q4_2024 = FiscalPeriod.quarter(2024, 4)
q1_2025 = FiscalPeriod.quarter(2025, 1)

print("Overlap tests:")
print(f"  FY 2024 overlaps FY 2025? {fy_2024.overlaps(fy_2025)} (should be False)")
print(f"  FY 2024 overlaps Q4 2024? {fy_2024.overlaps(q4_2024)} (should be True)")
print(f"  Q4 2024 overlaps Q1 2025? {q4_2024.overlaps(q1_2025)} (should be False)")

assert not fy_2024.overlaps(fy_2025), "Non-overlapping years detected as overlapping"
assert fy_2024.overlaps(q4_2024), "Year should overlap its own quarter"
assert not q4_2024.overlaps(q1_2025), "Adjacent quarters detected as overlapping"

print("\n✓ Overlap detection test passed")

Overlap tests:
  FY 2024 overlaps FY 2025? False (should be False)
  FY 2024 overlaps Q4 2024? True (should be True)
  Q4 2024 overlaps Q1 2025? False (should be False)

✓ Overlap detection test passed


### Test 6: Current and closed periods

In [40]:
# Create periods for testing
fy_2024 = FiscalPeriod.year(2024)
fy_2025 = FiscalPeriod.year(2025)
fy_2026 = FiscalPeriod.year(2026)

# Check against a reference date (today in notebook)
reference = date(2026, 2, 26)

print(f"Reference date: {reference}")
print(f"\n  FY 2024 is current? {fy_2024.is_current(reference)}")
print(f"  FY 2024 is closed? {fy_2024.is_closed(reference)}")
print(f"\n  FY 2025 is current? {fy_2025.is_current(reference)}")
print(f"  FY 2025 is closed? {fy_2025.is_closed(reference)}")
print(f"\n  FY 2026 is current? {fy_2026.is_current(reference)}")
print(f"  FY 2026 is closed? {fy_2026.is_closed(reference)}")

assert fy_2024.is_closed(reference), "FY 2024 should be closed"
assert fy_2025.is_closed(reference), "FY 2025 should be closed"
assert fy_2026.is_current(reference), "FY 2026 should be current"
assert not fy_2026.is_closed(reference), "FY 2026 should not be closed"

print("\n✓ Current/closed period test passed")

Reference date: 2026-02-26

  FY 2024 is current? False
  FY 2024 is closed? True

  FY 2025 is current? False
  FY 2025 is closed? True

  FY 2026 is current? True
  FY 2026 is closed? False

✓ Current/closed period test passed


### Test 7: Create multiple fiscal years

In [41]:
# Helper for creating consecutive years
years = create_fiscal_years(2024, 2026)

print("Created fiscal years:")
for year in years:
    print(f"  {year.name:10} {year.start_date} to {year.end_date}")

assert len(years) == 3, "Should create 3 years"
assert years[0].name == "FY 2024"
assert years[2].name == "FY 2026"

print("\n✓ Multiple years creation test passed")

Created fiscal years:
  FY 2024    2024-01-01 to 2024-12-31
  FY 2025    2025-01-01 to 2025-12-31
  FY 2026    2026-01-01 to 2026-12-31

✓ Multiple years creation test passed


### Test 8: Find period for date

In [42]:
years = create_fiscal_years(2024, 2026)

test_dates = [
    (date(2024, 6, 15), "FY 2024"),
    (date(2025, 1, 1), "FY 2025"),
    (date(2026, 12, 31), "FY 2026"),
]

print("Period lookup tests:")
for test_date, expected_name in test_dates:
    period = get_period_for_date(test_date, years)
    print(f"  {test_date} → {period.name if period else 'None'}")
    assert period is not None, f"Should find period for {test_date}"
    assert period.name == expected_name, f"Wrong period for {test_date}"

# Test date outside all periods
outside_date = date(2027, 1, 1)
period = get_period_for_date(outside_date, years)
print(f"  {outside_date} → {period}")
assert period is None, "Should return None for date outside all periods"

print("\n✓ Period lookup test passed")

Period lookup tests:
  2024-06-15 → FY 2024
  2025-01-01 → FY 2025
  2026-12-31 → FY 2026
  2027-01-01 → None

✓ Period lookup test passed


### Test 9: ERPNext format conversion

In [43]:
fy_2024 = FiscalPeriod.year(2024)
erpnext_payload = fy_2024.to_erpnext_format()

print("ERPNext fiscal year format:")
for key, value in erpnext_payload.items():
    print(f"  {key}: {value}")

assert erpnext_payload["doctype"] == "Fiscal Year"
assert erpnext_payload["year"] == "FY 2024"
assert erpnext_payload["year_start_date"] == "2024-01-01"
assert erpnext_payload["year_end_date"] == "2024-12-31"

print("\n✓ ERPNext format test passed")

ERPNext fiscal year format:
  doctype: Fiscal Year
  year: FY 2024
  year_start_date: 2024-01-01
  year_end_date: 2024-12-31

✓ ERPNext format test passed


### Test 10: Wellness Centre migration periods

In [44]:
# Migration data spans March 2024 - February 2025
migration_start = date(2024, 3, 1)
migration_end = date(2025, 2, 28)

migration_period = FiscalPeriod.custom(
    migration_start,
    migration_end,
    "Migration Period"
)

print(f"Migration period: {migration_period}")
print(f"Duration: {migration_period.duration_days} days")

# Check which fiscal years are needed
fy_2024 = FiscalPeriod.year(2024)
fy_2025 = FiscalPeriod.year(2025)

print(f"\nFiscal years needed for migration:")
print(f"  {fy_2024.name}: {fy_2024.overlaps(migration_period)}")
print(f"  {fy_2025.name}: {fy_2025.overlaps(migration_period)}")

assert fy_2024.overlaps(migration_period), "FY 2024 should overlap migration"
assert fy_2025.overlaps(migration_period), "FY 2025 should overlap migration"

# Sample transaction dates from wellness data
sample_dates = [
    date(2024, 3, 15),   # First month
    date(2024, 12, 31),  # Year end
    date(2025, 1, 1),    # New year
    date(2025, 2, 15),   # Last month
]

print(f"\nSample transaction date validation:")
for txn_date in sample_dates:
    in_migration = migration_period.contains(txn_date)
    fiscal_year = get_period_for_date(txn_date, [fy_2024, fy_2025])
    print(f"  {txn_date}: migration={in_migration}, FY={fiscal_year.name}")
    assert in_migration, f"{txn_date} should be in migration period"
    assert fiscal_year is not None, f"{txn_date} should have fiscal year"

print("\n✓ Wellness Centre migration periods test passed")

Migration period: Migration Period (2024-03-01 to 2025-02-28)
Duration: 365 days

Fiscal years needed for migration:
  FY 2024: True
  FY 2025: True

Sample transaction date validation:
  2024-03-15: migration=True, FY=FY 2024
  2024-12-31: migration=True, FY=FY 2024
  2025-01-01: migration=True, FY=FY 2025
  2025-02-15: migration=True, FY=FY 2025

✓ Wellness Centre migration periods test passed


## Summary - FiscalPeriod Class

**FiscalPeriod class validation complete:**
- ✓ Creates years, quarters, months
- ✓ Validates date ranges
- ✓ Checks date containment
- ✓ Detects period overlaps
- ✓ Identifies current/closed periods
- ✓ Finds periods for dates
- ✓ ERPNext format conversion
- ✓ Migration period validation

## Layer 1 Complete!

**All core types validated:**
- ✓ Money — Currency amounts with Decimal precision
- ✓ Account — Chart of Accounts with debit/credit logic
- ✓ Tax — Tax calculations with proper rounding
- ✓ FiscalPeriod — Accounting periods with date validation

**Next:** Extract stable code to `src/core/` and move to Layer 2 (GL Foundation)

In [46]:
# Test everything
#python3 << 'PYTEST'
import sys
sys.path.insert(0, 'src')

from core import Money, Account, AccountType, TaxRate, FiscalPeriod
from decimal import Decimal
from datetime import date

# Quick validation
m = Money(100, "KES")
acc = Account("Cash - WC", AccountType.CASH)
tax = TaxRate(Decimal('0.16'), "VAT @ 16%")
fy = FiscalPeriod.year(2024)

print("✓ Money:", m)
print("✓ Account:", acc)
print("✓ Tax:", tax)
print("✓ FiscalPeriod:", fy)
print("\n✓✓✓ Layer 1 complete!")
#PYTEST

✓ Money: KES 100.00
✓ Account: Cash - WC
✓ Tax: VAT @ 16% (16.00%)
✓ FiscalPeriod: FY 2024 (2024-01-01 to 2024-12-31)

✓✓✓ Layer 1 complete!


In [47]:
# These types now work together
base = Money(10000, "KES")
vat = TaxRate(Decimal('0.16'), "VAT @ 16%")
total = vat.calculate_total(base)
# → Money(11600.00, 'KES')

# Account knows its behavior
cash = Account("Cash - WC", AccountType.CASH)
assert cash.is_debit_positive()  # Debits increase cash

# Period validates dates
fy_2024 = FiscalPeriod.year(2024)
assert fy_2024.contains(date(2024, 6, 15))

# This will work in next layer
entry = JournalEntry([
    Line(account=cash, debit=total),
    Line(account=revenue, credit=base),
    Line(account=vat_account, credit=vat.calculate_tax(base))
])
# ← Validates debits = credits automatically